# Dual-GPU RAD-DINO embedding on Kaggle

Attach two private Kaggle Datasets before running. The source Dataset must contain `config/`, `scripts/`, `src/`, and `requirements.txt` at its root; exclude `.env`, `.venv-*`, `.hf-cache`, and `data/`. The processed Dataset must contain `images/` plus `metadata.jsonl` from `data/reprocessed`. Enable GPU and Internet. The notebook uses two independent CUDA processes, one per GPU, then merges their output into one FAISS index.

Do not upload `.env`. Store `HF_TOKEN` as a Kaggle Secret named `HF_TOKEN`.

In [ ]:
from pathlib import Path
import os
import shutil

# Change these to the slugs of the two attached Kaggle Datasets.
PROJECT_SOURCE_INPUT = Path('/kaggle/input/image-rag-source')
PROCESSED_INPUT = Path('/kaggle/input/iu-xray-reprocessed')
WORKING_DIR = Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'image-rag'
SHARD_DIR = WORKING_DIR / 'embedding-shards'
OUTPUT_DIR = WORKING_DIR / 'embeddings'
BATCH_SIZE_PER_GPU = 16

assert PROJECT_SOURCE_INPUT.is_dir(), PROJECT_SOURCE_INPUT
assert (PROCESSED_INPUT / 'images').is_dir(), PROCESSED_INPUT / 'images'
assert (PROCESSED_INPUT / 'metadata.jsonl').is_file(), PROCESSED_INPUT / 'metadata.jsonl'

shutil.rmtree(REPO_DIR, ignore_errors=True)
shutil.copytree(PROJECT_SOURCE_INPUT, REPO_DIR)
os.chdir(REPO_DIR)
print(f'Project: {REPO_DIR}')
print(f'Images: {PROCESSED_INPUT / "images"}')

In [ ]:
import subprocess
import sys
import torch

# Keep Kaggle's CUDA-enabled torch/torchvision wheels.
requirements = (REPO_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
non_torch_requirements = [
    line for line in requirements
    if not line.strip().startswith(('torch', 'torchvision'))
]
requirements_path = WORKING_DIR / 'requirements-kaggle.txt'
requirements_path.write_text('\n'.join(non_torch_requirements) + '\n', encoding='utf-8')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)])

assert torch.cuda.is_available(), 'Kaggle GPU accelerator is not enabled.'
assert torch.cuda.device_count() >= 2, (
    f'This notebook needs two GPUs, but Kaggle exposed {torch.cuda.device_count()}.'
)
for gpu_id in range(torch.cuda.device_count()):
    print(gpu_id, torch.cuda.get_device_name(gpu_id))

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

# Download the model once before both worker processes read the Hugging Face cache.
from src.embeddings.encoder import DEFAULT_MODEL_NAME, load_encoder, load_image_processor
load_image_processor(DEFAULT_MODEL_NAME)
warmup_model = load_encoder(device='cpu', model_name=DEFAULT_MODEL_NAME)
del warmup_model
print(f'Cached {DEFAULT_MODEL_NAME}')

In [ ]:
from src.data.metadata import read_metadata_jsonl, write_metadata_jsonl

rows = list(read_metadata_jsonl(PROCESSED_INPUT / 'metadata.jsonl'))
assert rows, 'No metadata rows found.'
assert len({row['id'] for row in rows}) == len(rows), 'Duplicate image IDs found.'

shutil.rmtree(SHARD_DIR, ignore_errors=True)
SHARD_DIR.mkdir(parents=True)
shard_paths = []
for gpu_id in range(2):
    shard_path = SHARD_DIR / f'metadata-gpu{gpu_id}.jsonl'
    # Interleaving balances varied X-ray aspect ratios across both GPUs.
    write_metadata_jsonl(rows[gpu_id::2], shard_path)
    shard_paths.append(shard_path)
    print(f'GPU {gpu_id}: {len(rows[gpu_id::2])} images')

In [ ]:
import subprocess

worker_dir = WORKING_DIR / 'embedding-workers'
shutil.rmtree(worker_dir, ignore_errors=True)
worker_dir.mkdir(parents=True)
workers = []

for gpu_id, shard_path in enumerate(shard_paths):
    worker_output = worker_dir / f'gpu{gpu_id}'
    log_path = worker_dir / f'gpu{gpu_id}.log'
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['PYTHONUNBUFFERED'] = '1'
    command = [
        sys.executable, 'scripts/compute_image_embeddings.py',
        '--processed_dir', str(PROCESSED_INPUT),
        '--metadata_path', str(shard_path),
        '--out_dir', str(worker_output),
        '--vector_db', 'none',
        '--device', 'cuda',
        '--batch_size', str(BATCH_SIZE_PER_GPU),
        '--clean',
    ]
    log_handle = log_path.open('w', encoding='utf-8')
    process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=log_handle, stderr=subprocess.STDOUT)
    workers.append((gpu_id, process, log_handle, log_path, worker_output))

for gpu_id, process, log_handle, log_path, _ in workers:
    exit_code = process.wait()
    log_handle.close()
    if exit_code:
        print(log_path.read_text(encoding='utf-8'))
        raise RuntimeError(f'GPU {gpu_id} worker failed with exit code {exit_code}.')
    print(f'GPU {gpu_id} completed. Log: {log_path}')

In [ ]:
import json
import faiss
import numpy as np

vectors_by_id = {}
manifest = None
for _, _, _, _, worker_output in workers:
    vectors = np.load(worker_output / 'embeddings.npy')
    worker_ids = (worker_output / 'ids.txt').read_text(encoding='utf-8').splitlines()
    assert len(vectors) == len(worker_ids)
    assert not (set(worker_ids) & set(vectors_by_id)), 'A shard produced duplicate IDs.'
    vectors_by_id.update(zip(worker_ids, vectors))
    if manifest is None:
        manifest = json.loads((worker_output / 'embedding_manifest.json').read_text(encoding='utf-8'))

ordered_ids = [row['id'] for row in rows]
assert set(ordered_ids) == set(vectors_by_id), 'Merged vectors do not match source metadata.'
embeddings = np.ascontiguousarray(np.vstack([vectors_by_id[item_id] for item_id in ordered_ids]), dtype=np.float32)

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.mkdir(parents=True)
np.save(OUTPUT_DIR / 'embeddings.npy', embeddings)
(OUTPUT_DIR / 'ids.txt').write_text('\n'.join(ordered_ids) + '\n', encoding='utf-8')
write_metadata_jsonl(rows, OUTPUT_DIR / 'metadata.jsonl')
(OUTPUT_DIR / 'embedding_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(OUTPUT_DIR / 'faiss.index'))
print(f'Merged {len(ordered_ids)} embeddings with dimension {embeddings.shape[1]}.')

In [ ]:
archive_path = shutil.make_archive(str(WORKING_DIR / 'image_embeddings'), 'zip', root_dir=OUTPUT_DIR)
print(f'Download this Kaggle output: {archive_path}')